# 01 — EmotionPlex dataset: inspect, validate, manifest

- Parses every sidecar JSON, validates the schema (prints samples so YOU can eyeball them).
- Agent reply label = `styletalk.res_emotion` (cross-checked against path / text_prompt).
- Agent turn spans from `turns[]`; word alignments for the text stream; `user_id` for grouped splits.
- Splits: official val manifest → `val`; 10% of train users → `holdout` (speaker generalization).
- Writes `manifest.parquet` (alignments/turns as JSON strings). CPU only.


In [ ]:
from pathlib import Path
import os, sys, json, time
import numpy as np
import torch

sys.path.insert(0, os.getcwd())
import eh_common as eh

P = eh.get_paths()   # override: P = eh.get_paths("/your/root") or set EMOTION_HEAD_ROOT
eh.mode_banner(P)    # FULL vs SMOKE_TEST — set once in NB00 (run_config.json)
RUN = eh.run_profile(P)   # all size knobs (full | smoke) — see eh_common.run_profile
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
print("ROOT:", P.ROOT, "| device:", DEVICE)
for k in ("MOSHI_BNB4", "MIMI_FILE", "SPM_FILE", "VOICE_PT"):
    p = Path(getattr(P, k))
    print(f"  {k}: {'OK' if p.exists() else 'MISSING'}  {p}")


In [ ]:
jsonl_files = sorted(P.DATA.rglob("*.jsonl"))
sidecars = [p for p in sorted(P.DATA.rglob("*.json"))
            if not p.name.startswith("_") and p.parent != P.DATA]
wav_by_name = {w.name: w for w in P.DATA.rglob("*.wav")}
print(f"{len(jsonl_files)} manifests | {len(sidecars)} sidecars | {len(wav_by_name)} wav")
rows = []
for jf in jsonl_files:
    for line in open(jf, encoding="utf-8"):
        line = line.strip()
        if line:
            r = json.loads(line)
            r["_manifest"] = jf.name
            rows.append(r)
print("manifest rows:", len(rows))
print("row keys:", sorted(rows[0].keys()))
print("example row:", {k: rows[0][k] for k in sorted(rows[0])})
if P.SMOKE:
    # keep only the stratified subset (deterministic, on disk). NB00 writes it
    # during the smoke download; if the dataset was already fully downloaded
    # (subset file missing), eh.ensure_smoke_subset derives it from the local
    # manifests instead — no re-download needed.
    keep = eh.ensure_smoke_subset(P, RUN, rows)
    rows = [r for r in rows if eh.manifest_clip_stem(r) in keep]
    print(f"SMOKE: restricted to the {len(keep)}-clip subset -> {len(rows)} rows")


In [ ]:
from collections import Counter
sidecar_by_name = {p.name: p for p in sidecars}
sample = json.loads(sidecars[0].read_text(encoding="utf-8"))
print("== sidecar example:", sidecars[0].name)
print(json.dumps(sample, indent=2, ensure_ascii=False)[:3000])
keysets = Counter()
for p in sidecars[:300]:
    try:
        keysets[tuple(sorted(json.loads(p.read_text(encoding="utf-8")).keys()))] += 1
    except Exception:
        pass
print("== sidecar key sets (top 5):")
for ks, n in keysets.most_common(5):
    print(f"  {n:5d}  {ks}")


In [ ]:
import re
import pandas as pd

AGENT_NAMES = {"agent", "assistant", "moshi", "bot", "personaplex",
               "speaker_broker", "speaker_0", "spk0", "left", "0"}
USER_NAMES = {"user", "human", "caller", "speaker_client",
              "speaker_1", "spk1", "right", "1"}

def find_emotion(d, path):
    st = d.get("styletalk")
    if isinstance(st, dict) and "res_emotion" in st:
        return str(st["res_emotion"])
    for k in ("styletalk.res_emotion", "res_emotion", "emotion", "label"):
        if k in d:
            return str(d[k])
    tp = str(d.get("text_prompt", "")).lower()
    for e in eh.EMOTIONS:
        if f"a {e} tone" in tp:
            return e
    m = re.search(r"(happy|sad|angry|neutral)", str(path).lower())
    return m.group(1) if m else None

def find_turns(d):
    out = []
    for t in (d.get("turns") or d.get("conversation") or []):
        if not isinstance(t, dict):
            continue
        # the dataset's `role` ("user"/"agent") is authoritative; `speaker` is
        # SPEAKER_CLIENT/SPEAKER_BROKER (channel labels)
        spk = str(t.get("role") or t.get("speaker", t.get("channel", ""))).strip().lower()
        s = t.get("start", t.get("x0", t.get("begin")))
        e = t.get("end", t.get("x1", t.get("stop")))
        if s is None or e is None:
            continue
        side = "agent" if spk in AGENT_NAMES else ("user" if spk in USER_NAMES else spk or "?")
        out.append((side, float(s), float(e)))
    return out

def find_alignments(d):
    out = []
    for w in (d.get("alignments") or d.get("alignment") or []):
        if isinstance(w, dict):
            out.append({k: w[k] for k in ("word", "start", "end", "speaker") if k in w})
        elif isinstance(w, (list, tuple)) and len(w) >= 2 and isinstance(w[1], (list, tuple)):
            rec = {"word": w[0], "start": float(w[1][0]), "end": float(w[1][1])}
            if len(w) > 2:
                rec["speaker"] = str(w[2])
            out.append(rec)
    return out

def guess_user_emotion(row, d):
    for k in ("user_emotion", "user_style"):
        if d.get(k):
            return str(d[k]).lower()
    return None

records, problems = [], []
for r in rows:
    path = str(r.get("path", ""))
    stem = Path(path).stem
    wav = wav_by_name.get(Path(path).name)
    sc = sidecar_by_name.get(stem + ".json")
    d = {}
    if sc is not None:
        try:
            d = json.loads(sc.read_text(encoding="utf-8"))
        except Exception:
            d = {}
    emo = str(find_emotion(d, path) or "").strip().lower()
    if emo not in eh.EMO2ID:
        problems.append((path, emo))
        continue
    turns = find_turns(d)
    agent_turns = [(s, e) for (side, s, e) in turns if side == "agent"]
    records.append(dict(
        clip_id=stem, rel_path=path,
        wav_path=str(wav) if wav else "", has_wav=bool(wav),
        manifest=r["_manifest"],
        duration=float(r.get("duration", 0.0) or 0.0),
        emotion=emo,
        user_id=str(d.get("user_id", r.get("user_id", ""))),
        user_emotion=guess_user_emotion(r, d),
        n_turns=len(turns), n_agent_turns=len(agent_turns),
        agent_turn_frames=json.dumps(
            [[int(round(s * eh.FRAME_HZ)), int(round(e * eh.FRAME_HZ))]
             for s, e in agent_turns]),
        alignments=json.dumps(find_alignments(d)),
    ))
print(f"records: {len(records)} | skipped: {len(problems)}")
if problems[:5]:
    print("skip examples:", problems[:5])
df = pd.DataFrame(records)
print(df["emotion"].value_counts())
print("agent turns per clip:", df.n_agent_turns.describe().round(2).to_dict())


In [ ]:
df["split"] = np.where(df["manifest"].str.lower().str.contains("val"), "val", "train")
train_users = np.array(sorted(df.loc[df.split == "train", "user_id"].unique()))
rng = np.random.RandomState(eh.SEED)
holdout = set(rng.permutation(train_users)[: max(1, len(train_users) // 10)])
df.loc[(df.split == "train") & df.user_id.isin(holdout), "split"] = "holdout"
P.MANIFEST.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(P.MANIFEST, index=False)   # full: <data>/manifest.parquet | SMOKE: _smoke/...
print(df.groupby(["split", "emotion"]).size().unstack(fill_value=0))
print(f"holdout users: {len(holdout)} | wav coverage: {df.has_wav.mean()*100:.1f}%")


In [ ]:
import subprocess
repo = P.ROOT / "refs" / "emotionplex"
if not repo.exists():
    repo.parent.mkdir(parents=True, exist_ok=True)
    for url in ("https://github.com/frabonzi/emotionplex",
                "https://github.com/mago-research/emotionplex"):
        r = subprocess.run(["git", "clone", "--depth", "1", url, str(repo)],
                           check=False, capture_output=True, text=True)
        if r.returncode == 0:
            print("cloned", url)
            break
        print("clone failed:", url)
il = repo / "finetune" / "finetune" / "data" / "interleaver.py"
if il.exists():
    print(il.read_text(encoding="utf-8", errors="replace")[:5000])
    print()
    print(">>> CROSS-CHECK: our text stream uses PAD=3 everywhere, EOP=0 one frame before")
    print(">>> each word, word pieces from the word-start frame (eh.build_agent_text_stream).")
    print(">>> If the interleaver above disagrees, fix eh_common.build_agent_text_stream")
    print(">>> BEFORE running notebook 02 (hidden states depend on the text stream).")
else:
    print("clone failed — check the interleaver convention manually before running 02")


In [ ]:
chk = pd.read_parquet(P.MANIFEST)
assert set(chk.emotion.unique()) <= set(eh.EMOTIONS)
assert chk.clip_id.is_unique
assert (chk.n_agent_turns >= 1).mean() > 0.95, "many clips without agent turns - check find_turns"
print("manifest.parquet OK:", len(chk), "clips |", chk.split.value_counts().to_dict())
print("-> continue with 02_extract_features_emotionplex.ipynb")
